<a href="https://colab.research.google.com/github/Naylet92/Estudio_ambiental/blob/main/Chamela_L_2000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Selección y procesamiento de imágenes Landsat (año 2000)

##Definir Variables

In [ ]:
# Prefijo del área
PREFIJO     = 'XMLA'
# Nombre del área
NOMBRE      = 'Chamela'

# Carpeta de salida en Drive
RUTA_AOI    = '/content/drive/MyDrive/ANP/AOI'

# Rutas derivadas automáticamente (no tocar)
RUTA        = f'{RUTA_AOI}/aoi_{PREFIJO}.geojson'
RUTA_GEOJSON= RUTA
RUTA_CSV    = f'{RUTA_AOI}/{PREFIJO}_coordenadas.csv'

# CRS geográfico y UTM
CRS_GEO     = 4326
CRS_UTM     = 32613

# Proyecto de Google Earth Engine
GEE_PROJECT = 'ee-nayleths'

AÑO           = 2000
COLECCION     = 'LANDSAT/LT05/C02/T1_L2'   # Landsat 5 TM para el año 2000

# Temporadas de análisis
TEMPORADA_SECA   = ('03-01', '05-31')        # marzo – mayo
TEMPORADA_HUMEDA = ('06-01', '12-31')        # agosto – noviembre

# Bandas espectrales de reflectancia superficial (Landsat 5)
BANDAS_SR     = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
BANDAS_CORR   = [b + '_corr' for b in BANDAS_SR]

# Coeficientes Tasseled Cap (Landsat 5 TM)
TC_BRIGHT = [0.3037, 0.2793, 0.4743, 0.5585, 0.5082, 0.1863]
TC_GREEN  = [-0.2848, -0.2435, -0.5436, 0.7243, 0.0840, -0.1800]
TC_WET    = [0.1509, 0.1973, 0.3279, 0.3406, -0.7112, -0.4572]

# Escala de exportación (m)
ESCALA        = 30

# Zoom inicial del mapa
ZOOM        = 12

# Estilo del área de estudio
STYLE_AREA  = {'color': 'blue', 'fillColor': '#0000ff30', 'weight': 1.5}

# Estilo del rectángulo rojo (bounding box)
STYLE_BBOX  = {'color': 'red', 'fillColor': '#00000000', 'weight': 2.5}

# Título del mapa
MAP_TITLE   = 'Área de estudio'

# Carpeta de salida en Drive para imágenes GEE
RUTA_IMAGENES = f'/content/drive/MyDrive/ANP/Landsat'

In [ ]:
import ee
import os
import math
import json
import geemap
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from google.colab import drive
from IPython.display import display, HTML

# Montar Google Drive
drive.mount('/content/drive')

# Autenticar e inicializar GEE
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

print("✓ Entorno listo")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Entorno listo


In [ ]:
# ─── FUNCIÓN: Enmascarar nubes y sombras ──────────────────────────────────────
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    cloud_shadow_mask = qa.bitwiseAnd(1 << 3).eq(0)
    cloud_mask        = qa.bitwiseAnd(1 << 4).eq(0)
    return image.updateMask(cloud_shadow_mask.And(cloud_mask))


def correccion_topografica(image, region):
    dem        = ee.Image('USGS/SRTMGL1_003').clip(region)
    terreno    = ee.Terrain.products(dem)

    pendiente   = terreno.select('slope').multiply(math.pi / 180)
    orientacion = terreno.select('aspect').multiply(math.pi / 180)

    az_solar   = ee.Number(image.get('SUN_AZIMUTH')).multiply(math.pi / 180)
    elev_solar = ee.Number(image.get('SUN_ELEVATION')).multiply(math.pi / 180)

    cos_i = (
        pendiente.cos().multiply(elev_solar.sin())
        .add(
            pendiente.sin()
            .multiply(elev_solar.cos())
            .multiply(orientacion.subtract(az_solar).cos())
        )
    )
    cos_i_safe = cos_i.max(0.2)

    # ← réplicar cos_i_safe para cada banda y construir imagen completa de golpe

    im_bandas = image.select(BANDAS_SR).multiply(0.0000275).add(-0.2)

    bandas_corr = (
    ee.Image.cat([
        im_bandas.select(b).divide(cos_i_safe).rename(BANDAS_CORR[i])
        for i, b in enumerate(BANDAS_SR)
        ])
        .set('SUN_AZIMUTH',   image.get('SUN_AZIMUTH'))
        .set('SUN_ELEVATION', image.get('SUN_ELEVATION'))
        .cast({b: 'double' for b in BANDAS_CORR})  # ← fuerza tipo homogéneo
    )

    return bandas_corr


def tasseled_cap(image_corr, coefs, nombre):
    partes = ee.Image.cat([
        image_corr.select(b).multiply(c)
        for b, c in zip(BANDAS_CORR, coefs)
    ])
    return partes.reduce(ee.Reducer.sum()).rename(nombre)

In [ ]:
# Región GEE desde el GeoJSON generado
gdf = gpd.read_file(RUTA_GEOJSON)
geojson_str = gdf.to_crs(epsg=4326).to_json()
region = ee.FeatureCollection(json.loads(geojson_str)).geometry()

def procesar_temporada(nombre_temporada, fecha_ini, fecha_fin):
    print(f"\n── Temporada: {nombre_temporada} ({fecha_ini} → {fecha_fin}) ──")

    fecha_inicio = f'{AÑO}-{fecha_ini}'
    fecha_fin_   = f'{AÑO}-{fecha_fin}'

    col = (ee.ImageCollection(COLECCION)
             .filterDate(fecha_inicio, fecha_fin_)
             .filterBounds(region))

    n = col.size().getInfo()
    print(f"  Imágenes encontradas: {n}")

    fechas = col.aggregate_array('DATE_ACQUIRED').getInfo()
    for f in sorted(fechas):
        print(f"    · {f}")

    if n == 0:
        print("  Sin imágenes para esta temporada.")
        return None

    # 1. Enmascarar nubes por imagen
    col_limpia = col.map(mask_clouds)

    # 2. Corrección topográfica por imagen
    col_corr = col_limpia.map(lambda img: correccion_topografica(img, region))

    # 3. Forzar bandas homogéneas antes del mosaico
    col_corr = col_corr.select(BANDAS_CORR)

    # 4. Mosaico de mediana
    mosaico = col_corr.median().clip(region)

    # 5. NDVI
    nir  = mosaico.select('SR_B4_corr')
    red  = mosaico.select('SR_B3_corr')
    ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

    # 6. Tasseled Cap
    tc_b = tasseled_cap(mosaico, TC_BRIGHT, 'TC_Brightness')
    tc_g = tasseled_cap(mosaico, TC_GREEN,  'TC_Greenness')
    tc_w = tasseled_cap(mosaico, TC_WET,    'TC_Wetness')

    resultado = (ee.Image.cat([ndvi, tc_b, tc_g, tc_w])
                   .clip(region)
                   .cast({'NDVI': 'double', 'TC_Brightness': 'double',
                          'TC_Greenness': 'double', 'TC_Wetness': 'double'}))
    print(f"  Bandas resultantes: {resultado.bandNames().getInfo()}")
    return resultado

### Exportar resultados a Google Drive

In [ ]:
# ─── Ejecutar procesamiento ───────────────────────────────────────────────────
img_seca   = procesar_temporada('Seca',   *TEMPORADA_SECA)
img_humeda = procesar_temporada('Húmeda', *TEMPORADA_HUMEDA)

# ─── Exportar a Google Drive ──────────────────────────────────────────────────
def exportar(imagen, sufijo):
    if imagen is None:
        print(f"Exportación cancelada para '{sufijo}': imagen no disponible.")
        return

    nombre_tarea = f'{PREFIJO}_{AÑO}_{sufijo}'
    tarea = ee.batch.Export.image.toDrive(
        image          = imagen,
        description    = nombre_tarea,
        folder         = 'Chamela_Landsat',
        fileNamePrefix = nombre_tarea,
        region         = region,
        scale          = ESCALA,
        crs            = f'EPSG:{CRS_UTM}',
        maxPixels      = 1e13
    )
    tarea.start()
    print(f"✓ Tarea iniciada: {nombre_tarea}")

exportar(img_seca,   'seca')
exportar(img_humeda, 'humeda')

### Visualización de imágenes procesadas

In [ ]:
stats = img_seca.select('NDVI').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_seca.select('TC_Brightness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_seca.select('TC_Greenness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_seca.select('TC_Wetness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)
#----------------Húmdad----------------------------------

stats = img_humeda.select('NDVI').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_humeda.select('TC_Brightness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_humeda.select('TC_Greenness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

stats = img_humeda.select('TC_Wetness').reduceRegion(
    reducer  = ee.Reducer.minMax(),
    geometry = region,
    scale    = 30,
    maxPixels= 1e9
).getInfo()
print(stats)

{'NDVI_max': 0.7307934921031648, 'NDVI_min': -0.10766347355268639}
{'TC_Brightness_max': 0.7910978126713324, 'TC_Brightness_min': 0.0013812376211188903}
{'TC_Greenness_max': 0.23038502043813322, 'TC_Greenness_min': -0.13828151476922218}
{'TC_Wetness_max': 0.2828180343900044, 'TC_Wetness_min': -0.3305710376479347}
{'NDVI_max': 0.8788218336267246, 'NDVI_min': -0.21375010735162195}
{'TC_Brightness_max': 1.008240366396126, 'TC_Brightness_min': 0.0014287946992033846}
{'TC_Greenness_max': 0.514133245787219, 'TC_Greenness_min': -0.3352651854418384}
{'TC_Wetness_max': 0.35532439261230714, 'TC_Wetness_min': -0.27007000427718875}


In [ ]:
# PARÁMETROS DE VISUALIZACIÓN

# Temporada SECA (basado en valores calculados)
VIS_SECA = {
    'NDVI': {'min': -0.10, 'max': 0.73, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 0.80, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.13, 'max': 0.23, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.33, 'max': 0.30, 'palette': ['brown', 'white', 'blue']}
}

# Temporada HÚMEDA (basado en valores calculados)
VIS_HUMEDA = {
    'NDVI': {'min': -0.13, 'max': 0.90, 'palette': ['#d73027', '#fee08b', '#1a9850']},
    'TC_Brightness': {'min': 0.00, 'max': 2.72, 'palette': ['white', 'brown']},
    'TC_Greenness': {'min': -0.45, 'max': 0.66, 'palette': ['white', 'darkgreen']},
    'TC_Wetness': {'min': -0.26, 'max': 0.19, 'palette': ['brown', 'white', 'blue']}
}

# Recalcular centroide
gdf = gpd.read_file(RUTA_GEOJSON)
centroid_proj = gdf.to_crs(epsg=3857).dissolve().centroid.iloc[0]
centroid = gpd.GeoSeries([centroid_proj], crs=3857).to_crs(epsg=4326).iloc[0]

#  MAPA
mapa2 = geemap.Map()
mapa2.set_center(centroid.x, centroid.y, ZOOM)

mapa2.add_gdf(gdf, layer_name='Área de estudio', style=STYLE_AREA)

if img_seca is not None:
    mapa2.addLayer(img_seca.select('NDVI'),          VIS_SECA['NDVI'], f'{AÑO} Seca – NDVI',          shown=True)
    mapa2.addLayer(img_seca.select('TC_Brightness'), VIS_SECA['TC_Brightness'],  f'{AÑO} Seca – TC Brightness',  shown=False)
    mapa2.addLayer(img_seca.select('TC_Greenness'),  VIS_SECA['TC_Greenness'],  f'{AÑO} Seca – TC Greenness',   shown=False)
    mapa2.addLayer(img_seca.select('TC_Wetness'),    VIS_SECA['TC_Wetness'],  f'{AÑO} Seca – TC Wetness',     shown=False)

if img_humeda is not None:
    mapa2.addLayer(img_humeda.select('NDVI'),          VIS_HUMEDA['NDVI'], f'{AÑO} Húmeda – NDVI',          shown=False)
    mapa2.addLayer(img_humeda.select('TC_Brightness'), VIS_HUMEDA['TC_Brightness'],  f'{AÑO} Húmeda – TC Brightness',  shown=False)
    mapa2.addLayer(img_humeda.select('TC_Greenness'),  VIS_HUMEDA['TC_Greenness'],  f'{AÑO} Húmeda – TC Greenness',   shown=False)
    mapa2.addLayer(img_humeda.select('TC_Wetness'),    VIS_HUMEDA['TC_Wetness'],  f'{AÑO} Húmeda – TC Wetness',     shown=False)

mapa2.addLayerControl()
print("Visualización de imágenes procesadas")
mapa2